# Ленивый Arsenal

Ноутбук использует две готовые конфигурации: многомодельную `test_playbook_arsenal_router_mode.toml` для Router Mode и `test_playbook_arsenal_model_mode.toml` с одной моделью на сервер для Model Mode. Конструктор только читает TOML и создаёт объектное дерево; ресурсы активируются при первом обращении к модели. Закомментированный метод `arsenal.download()` позволяет при желании заранее скачать все ресурсы без запуска процессов.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / ".zemicomp").is_file() and (directory / "zemi").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from zemi.playbook import Arsenal

## Построение объектного дерева

Следующие ячейки только читают TOML и создают объекты. Они не скачивают модели, не скачивают llama.cpp и не запускают процессы.

In [ ]:
router_mode_arsenal = Arsenal(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml"
)
# router_mode_arsenal.download()  # Предварительно скачать все ресурсы.

In [ ]:
model_mode_arsenal = Arsenal(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml"
)
# model_mode_arsenal.download()  # Предварительно скачать все ресурсы.

# Begin и end playbook

`begin_playbook` включает ленивый режим и сам ничего не скачивает и не запускает. При `stop_arsenal_before_begin=True` он только останавливает прежние процессы. Скачивание и запуск происходят при первом обращении к конкретной модели.

### Model Mode: чистый запуск

Рекомендуемый вариант: сначала остановить возможные старые процессы Arsenal, затем обращаться только к нужным моделям и остановить запущенные серверы после playbook.

In [ ]:
model_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=False,
)

primary_model = model_mode_arsenal.llamas.primary.models.qwen
secondary_model = model_mode_arsenal.llamas.secondary.models.phi

# На этом месте обе модели скачаны и оба сервера готовы.

model_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

### Model Mode: запуск без предварительной остановки

Используйте только когда известно, что настроенные порты свободны. Завершение с `False` намеренно оставляет серверы работающими для следующего playbook; последняя строка показывает явную последующую очистку.

In [ ]:
model_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=False,
    llama_router_mode=False,
)

model_mode_arsenal.llamas.primary.models.qwen

# Серверы остаются доступны после завершения playbook.
model_mode_arsenal.end_playbook(stop_arsenal_after_end=False)

# Выполните позже, когда серверы больше не нужны.
model_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

## Router Mode: чистый запуск

При первом обращении запускается родительский router с INI-пресетом, в котором заранее указаны пути всех моделей сервера. Последующие модели скачиваются и загружаются через `/models/load` без перезапуска router.

In [ ]:
router_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=True,
)

### Объектная модель запущенного Arsenal

После `begin_playbook` включена ленивая активация, но серверы ещё не запущены. Первое обращение по цепочке `llamas → models` готовит выбранную модель. На каждом уровне объект можно получить по индексу, строковому имени или через точку; свойство `config` содержит исходную TOML-таблицу.

In [ ]:
# Llama-сервер: индекс, имя и точечная запись.
primary_by_index = router_mode_arsenal.llamas[0]
primary_by_name = router_mode_arsenal.llamas["primary"]
primary_by_dot = router_mode_arsenal.llamas.primary
assert primary_by_index is primary_by_name is primary_by_dot

# Модель: те же три способа доступа.
qwen_by_index = primary_by_dot.models[0]
qwen_by_name = primary_by_dot.models["qwen"]
qwen_by_dot = primary_by_dot.models.qwen
assert qwen_by_index is qwen_by_name is qwen_by_dot

# Ассистент: те же три способа доступа.
assistant_by_index = qwen_by_dot.assistants[0]
assistant_by_name = qwen_by_dot.assistants["assistant"]
assistant_by_dot = qwen_by_dot.assistants.assistant
assert assistant_by_index is assistant_by_name is assistant_by_dot

# Коллекции сохраняют порядок и поддерживают отрицательные индексы.
assert router_mode_arsenal.llamas[-1].name == "secondary"
assert primary_by_dot.models[-1].name == "smollm"

{
    "llama_names": list(router_mode_arsenal.llamas.keys()),
    "model_names": list(primary_by_dot.models.keys()),
    "assistant_names": list(qwen_by_dot.assistants.keys()),
    "llama_config": primary_by_dot.config,
    "model_config": qwen_by_dot.config,
    "assistant_config": assistant_by_dot.config,
}

### Завершение playbook

После демонстрации объектной модели останавливаем все серверы из конфигурации.

In [ ]:
router_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

## Router Mode: запуск на заведомо свободных портах

Вариант без предварительной остановки полезен, когда состояние окружения контролируется снаружи. Серверы останавливаются после playbook.

In [ ]:
router_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=False,
    llama_router_mode=True,
)

router_mode_arsenal.llamas.primary.models.qwen
router_mode_arsenal.llamas.primary.models.smollm
# Вторая модель добавлена без перезапуска родительского router.

router_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

## Проверка ограничения Model Mode

Исходный Arsenal содержит несколько моделей на сервер. Поэтому попытка запустить его без Router Mode ожидаемо завершается `ValueError` до запуска первого сервера.

In [ ]:
try:
    router_mode_arsenal.begin_playbook(
        stop_arsenal_before_begin=False,
        llama_router_mode=False,
    )
except ValueError as error:
    print(f"Ожидаемая ошибка конфигурации: {error}")